# 179 — Weight Space Geometry

Analyze the geometry of weight matrices: manifold dimension,
weight distances, symmetry analysis, interpolation effects,
and parameter norm geometry.

## Why This Matters

Weight space geometry examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_space_geometry import (
    weight_manifold_dimension,
    weight_distance_profile,
    weight_symmetry_analysis,
    weight_interpolation_effect,
    parameter_norm_geometry,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
for layer in range(4):
    result = weight_manifold_dimension(model, layer=layer)
    print(f"Layer {layer}:")
    for name, m in result['matrices'].items():
        print(f"  {name:5s}: eff_dim={m['effective_dimension']:.1f}/{m['full_rank']}  "
              f"util={m['rank_utilization']:.3f}  cond={m['condition_number']:.1f}")
    print()

In [ ]:
result = weight_distance_profile(model)
for name, d in result['distances'].items():
    print(f"{name}: mean_dist={d['mean_distance']:.4f}")
    for i, row in enumerate(d['distance_matrix']):
        vals = '  '.join(f'{v:.3f}' for v in row)
        print(f"  L{i}: [{vals}]")
    print()

In [ ]:
result = weight_symmetry_analysis(model, layer=0)
print(f"Global QK cosine: {result['global_qk_cosine']:.4f}\n")
for h in result['per_head']:
    print(f"Head {h['head']}: QK={h['qk_cosine']:.3f}  "
          f"QV={h['qv_cosine']:.3f}  KV={h['kv_cosine']:.3f}")

In [ ]:
result = weight_interpolation_effect(model, tokens, layer=1, matrix_name='W_in', n_steps=6)
print("Interpolating W_in toward zero:\n")
for step in result['interpolation']:
    bar = '#' * int(step['max_logit_change'] * 10)
    print(f"  alpha={step['alpha']:.2f}: max_change={step['max_logit_change']:.4f}  {bar}")

In [ ]:
result = parameter_norm_geometry(model)
print(f"Global: mean={result['global_mean_norm']:.4f}  "
      f"std={result['global_std_norm']:.4f}  "
      f"range=[{result['global_min_norm']:.4f}, {result['global_max_norm']:.4f}]")
print(f"Embed norm: {result['embed_norm']:.4f}  Unembed norm: {result['unembed_norm']:.4f}\n")
for layer in result['per_layer']:
    norms = '  '.join(f'{n}={v:.3f}' for n, v in layer['norms'].items())
    print(f"Layer {layer['layer']}: {norms}")